# Agente en Machine Learning con Scikit-Learn y TensorFlow
**RAG + LangGraph + Gemini + ChromaDB + Memoria de conversación**

Este notebook construye un agente de IA capaz de responder preguntas sobre el libro *Machine Learning with Scikit-Learn and TensorFlow* usando RAG, LangGraph como framework de agente, Google Gemini como LLM y embeddings, y ChromaDB como base de conocimiento vectorial.

In [1]:
# Victor
uuid_alumno="a197b4bd-b485-4e8d-94e1-b8f63daffd2e"

## Celda 1 — Instalación de dependencias

In [2]:
# Instalar dependencias (ejecutar solo una vez)
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
#     'langchain>=0.3.0',
#     'langchain-google-genai>=2.0.0',
#     'langchain-chroma>=0.1.0',
#     'langchain-text-splitters>=0.3.0',
#     'langgraph>=0.2.0',
#     'chromadb>=0.5.0',
#     'pypdf>=4.0.0',
#     'python-dotenv>=1.0.0',
#     'ipywidgets>=8.0.0',
# ])
print('✅ Dependencias ya instaladas')

✅ Dependencias ya instaladas


## Celda 2 — Imports y configuración de la API key

In [3]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Annotated
import operator

# Cargar variables de entorno desde .env
load_dotenv(Path('.env'))

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('No se encontró GEMINI_API_KEY en el archivo .env')

MODEL_NAME = 'gemini-2.5-flash-lite'
EMBEDDING_MODEL = 'models/gemini-embedding-001'
CHROMA_DIR = './chroma_ml_book'
PDF_PATH = './Machine Learning with Scikit Learn and TensorFlow.pdf'

print(f'✅ API key cargada. Modelo: {MODEL_NAME}')

✅ API key cargada. Modelo: gemini-2.5-flash-lite


## Celda 3 — Carga del PDF y limpieza del texto

In [4]:
def clean_text(text: str) -> str:
    """Limpieza básica del texto extraído del PDF."""
    # Eliminar encabezados/pies de página repetidos (líneas muy cortas o de solo números)
    lines = text.split('\n')
    cleaned = []
    for line in lines:
        stripped = line.strip()
        # Descartar líneas vacías, números de página solos y líneas de solo puntos/guiones
        if not stripped:
            continue
        if re.fullmatch(r'\d+', stripped):
            continue
        if re.fullmatch(r'[\-\.\|_]{3,}', stripped):
            continue
        cleaned.append(stripped)
    # Unir con espacio simple y colapsar espacios múltiples
    result = ' '.join(cleaned)
    result = re.sub(r' {2,}', ' ', result)
    return result.strip()


reader = PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f'📄 Páginas totales en el PDF: {total_pages}')

raw_pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text() or ''
    cleaned = clean_text(text)
    if cleaned:
        raw_pages.append({'page': i + 1, 'text': cleaned})

print(f'✅ Páginas con texto extraíble: {len(raw_pages)}')
print(f'\nEjemplo (página 1, primeros 400 chars):\n{raw_pages[0]["text"][:400]}')

📄 Páginas totales en el PDF: 564
✅ Páginas con texto extraíble: 564

Ejemplo (página 1, primeros 400 chars):
Aurélien Géron Hands-On Machine Learning with Scikit-Learn & TensorFlow CONCEPTS, TOOLS, AND TECHNIQUES TO BUILD INTELLIGENT SYSTEMS  D o w n l o a d   f r o m   f i n e l y b o o k   w w w . f i n e l y b o o k . c o m


## Celda 4 — Segmentación de texto (chunking) y construcción de documentos

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

documents = []
for page_info in raw_pages:
    chunks = splitter.split_text(page_info['text'])
    for chunk_idx, chunk in enumerate(chunks):
        # Filtrar chunks con ruido (menos de 60 caracteres útiles)
        if len(chunk.strip()) < 60:
            continue
        doc = Document(
            page_content=chunk.strip(),
            metadata={
                'page': page_info['page'],
                'chunk': chunk_idx,
                'source': PDF_PATH,
            }
        )
        documents.append(doc)

print(f'✅ {len(documents)} chunks generados desde {len(raw_pages)} páginas')
print(f'   Tamaño promedio de chunk: {sum(len(d.page_content) for d in documents) // len(documents)} chars')
print(f'\nEjemplo de chunk (doc 0):\n{documents[0].page_content[:300]}')

✅ 1830 chunks generados desde 564 páginas
   Tamaño promedio de chunk: 604 chars

Ejemplo de chunk (doc 0):
Aurélien Géron Hands-On Machine Learning with Scikit-Learn & TensorFlow CONCEPTS, TOOLS, AND TECHNIQUES TO BUILD INTELLIGENT SYSTEMS  D o w n l o a d   f r o m   f i n e l y b o o k   w w w . f i n e l y b o o k . c o m


## Celda 5 — Creación o carga de la base de conocimiento vectorial en ChromaDB

In [6]:
import time
import shutil
import chromadb

embedding_model = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=GEMINI_API_KEY
)

chroma_path = Path(CHROMA_DIR)

# Verificar si ya existe una colección válida en disco
_existing_count = 0
if chroma_path.exists():
    try:
        _client = chromadb.PersistentClient(path=CHROMA_DIR)
        _col = _client.get_or_create_collection('ml_book')
        _existing_count = _col.count()
        del _client, _col
    except Exception:
        _existing_count = 0

if _existing_count > 0:
    print(f'📂 Cargando vector store existente desde disco ({_existing_count} chunks)...')
    vectorstore = Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embedding_model,
        collection_name='ml_book'
    )
    print(f'✅ Vector store cargado. Chunks: {vectorstore._collection.count()}')
else:
    # Limpiar directorio incompleto si existe
    if chroma_path.exists():
        shutil.rmtree(chroma_path)
        print(f'🗑️  Directorio {CHROMA_DIR} eliminado (colección incompleta).')

    # Free tier: 100 req/min. BATCH=5 + SLEEP=65s → ~4.6 req/min (margen seguro).
    # Tiempo estimado para 1830 chunks: ~(1830/5) * 65s ≈ 40 min.
    BATCH = 5
    SLEEP_BETWEEN_BATCHES = 65  # segundos
    MAX_RETRIES = 5

    total = len(documents)
    print(f'⏳ Creando vector store con {total} chunks.')
    print(f'   Lote={BATCH} docs · Pausa={SLEEP_BETWEEN_BATCHES}s · ~{total//BATCH} lotes')
    print(f'   Tiempo estimado: ~{total // BATCH * SLEEP_BETWEEN_BATCHES // 60} minutos\n')

    vectorstore = None
    for i in range(0, total, BATCH):
        batch = documents[i:i + BATCH]

        # Reintentos con backoff exponencial ante errores transitorios (429, etc.)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                if vectorstore is None:
                    vectorstore = Chroma.from_documents(
                        documents=batch,
                        embedding=embedding_model,
                        persist_directory=CHROMA_DIR,
                        collection_name='ml_book'
                    )
                else:
                    vectorstore.add_documents(batch)
                break  # éxito
            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise
                wait = 2 ** attempt * 10  # 20s, 40s, 80s, 160s
                print(f'  ⚠️  Error en lote {i//BATCH+1}, reintento {attempt}/{MAX_RETRIES-1} en {wait}s... ({e})')
                time.sleep(wait)

        processed = i + len(batch)
        if processed % 100 == 0 or processed >= total:
            print(f'  ✔ {processed}/{total} chunks indexados...')
        time.sleep(SLEEP_BETWEEN_BATCHES)

    print(f'\n✅ Vector store creado con {vectorstore._collection.count()} chunks')

📂 Cargando vector store existente desde disco (100 chunks)...
✅ Vector store cargado. Chunks: 100


## Celda 6 — Verificación del vector store con consultas de prueba

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

test_queries = [
    'how does a Random Forest work',
    'training a neural network with TensorFlow',
    'overfitting and regularization techniques',
]

for q in test_queries:
    results = retriever.invoke(q)
    print(f'\n🔍 Query: "{q}"')
    for r in results[:2]:
        print(f'  → (p.{r.metadata.get("page", "?")}) {r.page_content[:130]}...')

print('\n✅ El vector store responde correctamente a consultas de prueba')


🔍 Query: "how does a Random Forest work"
  → (p.8) 6. Decision Trees. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . ....
  → (p.8) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 181 Voting Classifiers 181 Bagging and P asting 185 Bagg...

🔍 Query: "training a neural network with TensorFlow"
  → (p.9) . Introduction to Artificial Neural Networks. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 253 From Biologi...
  → (p.18) Part II, Neural Networks and Deep Learning, covers the following topics: • Wha t are neural nets? Wha t are they good for? • Build...

🔍 Query: "overfitting and regularization techniques"
  → (p.10) 303 Dropout 304 Max-Norm Regularization 307 Data Augmentation 309 Practical Guidelines 310 Exercises 311 12...
  → (p.10) . 275 Vanishing/Exploding Gradients Problems Xavier and He Initialization 277 Nonsaturating Activation Functions 279 Batch Normali...

✅ El 

## Celda 7 — System prompt y configuración del LLM

In [8]:
SYSTEM_PROMPT = """Eres un agente experto en Machine Learning con Python, especializado en los contenidos del libro "Machine Learning with Scikit-Learn and TensorFlow".
Tu base de conocimiento contiene el texto completo del libro: conceptos teóricos, algoritmos, implementaciones con Scikit-Learn y TensorFlow/Keras, y ejemplos de código en Python.

Rol y tono:
- Responde siempre en español, de forma clara, precisa y didáctica.
- Actúa como un instructor experimentado: explica los conceptos paso a paso, incluye ejemplos de código cuando sea útil, y señala las mejores prácticas.
- No inventes información. Si el contexto no contiene suficiente detalle, indícalo y sugiere en qué capítulo o sección podría encontrarse la respuesta.

Limitaciones:
- Solo puedes responder preguntas relacionadas con los temas del libro: algoritmos de ML, redes neuronales, preprocesamiento de datos, evaluación de modelos, Scikit-Learn, TensorFlow, Keras, etc.
- Si te preguntan algo fuera de ese dominio, declina amablemente y redirige al tema de Machine Learning.

Comportamiento cuando no hay información suficiente:
- Indica explícitamente que no encontraste esa información en el contexto recuperado.
- Sugiere reformular la pregunta o buscar con términos más específicos del libro.

Contexto recuperado del vector store:
{context}"""

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=GEMINI_API_KEY,
    temperature=0.2,
)

print('✅ LLM y system prompt configurados')

✅ LLM y system prompt configurados


## Celda 8 — Construcción del grafo LangGraph con RAG y memoria

In [12]:
# Estado del grafo
class AgentState(TypedDict):
    messages: Annotated[List, operator.add]   # historial completo de mensajes
    context: str                               # contexto recuperado de ChromaDB
    question: str                              # pregunta actual del usuario
    route: str                                 # 'RAG' o 'DIRECTO'


# --- Nodo 0: Decidir si hace falta recuperar contexto del libro ---
# Palabras clave que indican que la pregunta requiere conocimiento del libro.
RAG_KEYWORDS = [
    # Conceptos generales de ML
    'machine learning', 'aprendizaje automático', 'aprendizaje supervisado', 'no supervisado',
    'reinforcement learning', 'refuerzo', 'clasificación', 'regresión', 'clustering',
    'agrupamiento', 'overfitting', 'underfitting', 'sesgo', 'varianza', 'bias', 'variance',
    'precisión', 'accuracy', 'recall', 'exhaustividad', 'f1', 'matriz de confusión',
    'curva roc', 'auc', 'cross-validation', 'validación cruzada', 'hiperparámetro',
    'feature', 'característica', 'preprocesamiento', 'normalización', 'estandarización',
    # Algoritmos y modelos
    'regresión lineal', 'linear regression', 'logistic', 'logística', 'svm', 'máquina de vectores',
    'support vector', 'árbol', 'decision tree', 'random forest', 'gradient boosting', 'xgboost',
    'k-means', 'kmeans', 'k nearest', 'knn', 'naive bayes', 'perceptrón', 'red neuronal',
    'neural network', 'deep learning', 'cnn', 'rnn', 'transformer', 'autoencoder', 'gan',
    # Scikit-Learn / Keras / TensorFlow
    'scikit-learn', 'sklearn', 'tensorflow', 'keras', 'tf', 'pipeline', 'estimator',
    'fit', 'predict', 'modelo', 'entrenamiento', 'epochs', 'batch', 'optimizer',
    'loss', 'metric', 'callback', 'early stopping', 'dropout', 'regularización',
    'l1', 'l2', 'batch normalization', 'data augmentation', 'transfer learning',
    # Libro / capítulos
    'libro', 'capítulo', 'chapter', 'página', 'page', 'aurélien', 'geron',
]


def route(state: AgentState) -> dict:
    """Decide si es necesario recuperar contexto del libro."""
    question = state['question'].lower()
    needs_rag = any(kw in question for kw in RAG_KEYWORDS)
    return {'route': 'RAG' if needs_rag else 'DIRECTO'}


# --- Nodo 1: Recuperar contexto de ChromaDB ---
def retrieve(state: AgentState) -> dict:
    question = state['question']
    docs = retriever.invoke(question)
    context = '\n\n'.join([d.page_content for d in docs])
    return {'context': context}


# --- Nodo 2: Generar respuesta con Gemini ---
def generate(state: AgentState) -> dict:
    context = state.get('context', '')
    question = state['question']
    history = state.get('messages', [])

    if state.get('route') == 'DIRECTO':
        system_content = (
            "Eres un asistente amable experto en Machine Learning. "
            "Responde la pregunta del usuario de forma breve y cordial. "
            "Si el usuario pregunta sobre ML, invítalo a consultar el libro."
        )
    else:
        system_content = SYSTEM_PROMPT.format(context=context)

    # Construir la lista de mensajes: system + historial + pregunta actual
    prompt_messages = [SystemMessage(content=system_content)]
    prompt_messages.extend(history)
    prompt_messages.append(HumanMessage(content=question))

    response = llm.invoke(prompt_messages)

    # Guardar la pregunta y respuesta en el historial
    new_messages = [
        HumanMessage(content=question),
        AIMessage(content=response.content),
    ]
    return {'messages': new_messages}


# --- Construcción del grafo ---
workflow = StateGraph(AgentState)
workflow.add_node('route', route)
workflow.add_node('retrieve', retrieve)
workflow.add_node('generate', generate)

workflow.set_entry_point('route')
workflow.add_conditional_edges(
    'route',
    lambda state: state['route'],
    {'RAG': 'retrieve', 'DIRECTO': 'generate'}
)
workflow.add_edge('retrieve', 'generate')
workflow.add_edge('generate', END)

agent = workflow.compile()

print('✅ Grafo LangGraph compilado correctamente')

✅ Grafo LangGraph compilado correctamente


## Celda 9 — Función helper para conversar con el agente

In [10]:
# Estado de conversación persistente entre celdas
conversation_state: AgentState = {
    'messages': [],
    'context': '',
    'question': ''
}

def ask(question: str, verbose: bool = False) -> str:
    """Envía una pregunta al agente y retorna la respuesta. Mantiene el historial."""
    global conversation_state
    
    # Actualizar la pregunta manteniendo el historial acumulado
    input_state: AgentState = {
        'messages': conversation_state['messages'],
        'context': '',
        'question': question
    }
    
    result = agent.invoke(input_state)
    
    # Actualizar el estado de conversación global
    conversation_state['messages'] = result['messages']
    conversation_state['context'] = result['context']
    
    answer = result['messages'][-1].content
    
    if verbose:
        print(f'--- Contexto recuperado ---')
        print(result['context'][:500])
        print('---')
    
    print(f'\n👤 **Pregunta:** {question}')
    print(f'\n🤖 **Respuesta:**\n{answer}')
    print('\n' + '='*70)
    return answer


def reset_conversation():
    """Reinicia el historial de conversación."""
    global conversation_state
    conversation_state = {'messages': [], 'context': '', 'question': ''}
    print('🔄 Conversación reiniciada.')


print('✅ Funciones ask() y reset_conversation() disponibles')

✅ Funciones ask() y reset_conversation() disponibles


## Celda 10 — Ejemplos documentados (mínimo 5 preguntas)

In [13]:
reset_conversation()

# Pregunta 1: Concepto general
_ = ask('¿Qué es el aprendizaje supervisado y cómo se diferencia del no supervisado?')

🔄 Conversación reiniciada.

👤 **Pregunta:** ¿Qué es el aprendizaje supervisado y cómo se diferencia del no supervisado?

🤖 **Respuesta:**
El **aprendizaje supervisado** es una categoría de Machine Learning donde los datos de entrenamiento que se le proporcionan al algoritmo incluyen las **soluciones deseadas**, conocidas como **etiquetas**. Piensa en ello como aprender con un "maestro" que te dice la respuesta correcta para cada ejemplo.

Un ejemplo típico de aprendizaje supervisado es la **clasificación**. Imagina un filtro de spam: se entrena con muchos correos electrónicos junto con su etiqueta (si es "spam" o "no spam"). El objetivo del algoritmo es aprender a clasificar correos electrónicos nuevos.

Otra tarea común en aprendizaje supervisado es la **regresión**, donde se predice un valor numérico objetivo, como el precio de un coche, basándose en un conjunto de características (kilometraje, edad, marca, etc.).

Por otro lado, el **aprendizaje no supervisado**, como su nombre indi

In [14]:
# Pregunta 2: Algoritmo específico con código
_ = ask('¿Cómo funciona el algoritmo Random Forest y cómo se entrena con Scikit-Learn?')


👤 **Pregunta:** ¿Cómo funciona el algoritmo Random Forest y cómo se entrena con Scikit-Learn?

🤖 **Respuesta:**
El algoritmo **Random Forest** es una técnica de **aprendizaje ensamblado** muy potente y popular, que se basa en la idea de combinar múltiples modelos de árboles de decisión para obtener predicciones más precisas y robustas.

Aquí te explico cómo funciona y cómo se entrena con Scikit-Learn:

### ¿Cómo funciona Random Forest?

La idea central de Random Forest es la **diversidad de los árboles**. En lugar de entrenar un solo árbol de decisión, que puede ser propenso al sobreajuste (overfitting), Random Forest entrena un conjunto de árboles de decisión y luego combina sus predicciones. Para lograr esta diversidad, se aplican dos técnicas clave durante el entrenamiento de cada árbol:

1.  **Bagging (Bootstrap Aggregating):**
    *   Para entrenar cada árbol, se toma una **muestra aleatoria con reemplazo** del conjunto de datos de entrenamiento original. Esto significa que algun

In [15]:
# Pregunta 3: Evaluación de modelos
_ = ask('¿Qué métricas se usan para evaluar un modelo de clasificación y cuándo usar cada una?')


👤 **Pregunta:** ¿Qué métricas se usan para evaluar un modelo de clasificación y cuándo usar cada una?

🤖 **Respuesta:**
Para evaluar un modelo de clasificación, existen varias métricas importantes que nos ayudan a entender su rendimiento desde diferentes perspectivas. La elección de la métrica adecuada depende del problema específico y de las consecuencias de los diferentes tipos de errores.

Aquí te presento las métricas más comunes y cuándo usarlas:

### 1. Precisión (Accuracy)

*   **¿Qué es?** Es la métrica más intuitiva. Representa la proporción de predicciones correctas sobre el total de predicciones realizadas.
    $$ \text{Accuracy} = \frac{\text{Número de predicciones correctas}}{\text{Número total de predicciones}} = \frac{TP + TN}{TP + TN + FP + FN} $$
    Donde:
    *   `TP` (True Positives): Verdaderos Positivos (correctamente clasificados como positivos).
    *   `TN` (True Negatives): Verdaderos Negativos (correctamente clasificados como negativos).
    *   `FP` (False 

In [16]:
# Pregunta 4: Redes neuronales con TensorFlow/Keras
_ = ask('¿Cómo se construye y compila una red neuronal densa con Keras en TensorFlow?')


👤 **Pregunta:** ¿Cómo se construye y compila una red neuronal densa con Keras en TensorFlow?

🤖 **Respuesta:**
Construir y compilar una red neuronal densa (también conocida como red neuronal completamente conectada o *fully connected*) con Keras en TensorFlow es un proceso bastante directo. Keras proporciona una API de alto nivel que simplifica enormemente la creación de modelos.

Aquí te explico los pasos principales:

### 1. Importar las Librerías Necesarias

Primero, necesitas importar las capas y el modelo de Keras que vas a utilizar.

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
```

*   `tensorflow` y `keras`: El framework principal.
*   `Sequential`: Un contenedor para modelos que se componen de una pila lineal de capas. Es el tipo de modelo más común para redes neuronales densas.
*   `Dense`: Representa una capa densamente conectada, donde cada neurona de la capa está con

In [17]:
# Pregunta 5: Regularización y overfitting
_ = ask('¿Qué técnicas existen para evitar el overfitting en modelos de Machine Learning?')


👤 **Pregunta:** ¿Qué técnicas existen para evitar el overfitting en modelos de Machine Learning?

🤖 **Respuesta:**
El **sobreajuste (overfitting)** ocurre cuando un modelo aprende los datos de entrenamiento tan bien que empieza a memorizar el ruido y los detalles específicos de esos datos, en lugar de aprender los patrones generales. Como resultado, el modelo funciona muy bien en los datos de entrenamiento, pero su rendimiento decae drásticamente cuando se le presentan datos nuevos y no vistos.

Afortunadamente, existen varias técnicas para mitigar o evitar el sobreajuste. Aquí te presento algunas de las más importantes, muchas de las cuales se discuten en el libro:

### 1. Aumento de Datos (Data Augmentation)

*   **¿En qué consiste?** Consiste en crear nuevas muestras de entrenamiento a partir de las existentes mediante la aplicación de transformaciones aleatorias que preservan la etiqueta. Por ejemplo, en imágenes, se pueden aplicar rotaciones, desplazamientos, zoom, volteos, cambi

In [18]:
# Pregunta 6: Demostración de MEMORIA — referencia a respuesta anterior
_ = ask('De las técnicas que mencionaste para evitar overfitting, ¿cuál recomienda el libro para redes neuronales profundas?')


👤 **Pregunta:** De las técnicas que mencionaste para evitar overfitting, ¿cuál recomienda el libro para redes neuronales profundas?

🤖 **Respuesta:**
El libro recomienda varias técnicas para evitar el sobreajuste en redes neuronales profundas, y a menudo, la mejor estrategia es **combinar varias de ellas**. Sin embargo, algunas técnicas son particularmente destacadas y se discuten con mayor detalle en el contexto de redes profundas:

1.  **Regularización (L1, L2, Elastic Net):**
    *   El libro menciona explícitamente la regularización L1 y L2 como métodos para evitar el sobreajuste en el **Capítulo 11.2 "Avoiding Overfitting Through Regularization"**. Se explica que añadir un término de penalización a la función de pérdida basado en la magnitud de los pesos ayuda a mantener los pesos pequeños, lo que resulta en modelos más simples y generalizables. En Keras, esto se implementa a través de los argumentos `kernel_regularizer` y `bias_regularizer` en las capas.

2.  **Dropout:**
    * 

In [19]:
# Pregunta 7: Nueva demostración de MEMORIA — continuidad temática
_ = ask('¿Y cómo se implementa esa técnica en Keras? ¿Puedes mostrar el código de ejemplo?')


👤 **Pregunta:** ¿Y cómo se implementa esa técnica en Keras? ¿Puedes mostrar el código de ejemplo?

🤖 **Respuesta:**
¡Claro! Vamos a ver cómo implementar las técnicas más relevantes para evitar el sobreajuste en redes neuronales profundas usando Keras.

### 1. Dropout

Se implementa añadiendo una capa `Dropout` entre otras capas.

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l1, l2, l1_l2

# Ejemplo de modelo con Dropout
input_features = 20
num_classes = 1 # Para clasificación binaria

model_dropout = Sequential([
    # Capa oculta 1
    Dense(units=128, activation='relu', input_shape=(input_features,)),
    # Capa de Dropout después de la capa oculta 1
    # rate: la fracción de neuronas que se apagarán aleatoriamente. Un valor común es 0.2 a 0.5.
    Dropout(rate=0.3),

    # Capa oculta 2
    Dense(units=64, activation='relu'),
    # Capa de Dropout después de la capa oculta 2
  

## Celda 11 — Chat interactivo (conversación libre con el agente)

In [20]:
# Ejecuta esta celda para iniciar una conversación interactiva.
# Escribe 'salir' o 'exit' para terminar.
# Escribe 'reset' para limpiar el historial.

print('💬 Chat interactivo con el Agente Experto en ML con Scikit-Learn y TensorFlow')
print('Comandos especiales: "salir" / "exit" para terminar | "reset" para reiniciar historial')
print('='*70)

while True:
    try:
        user_input = input('\n👤 Tú: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n👋 Hasta luego.')
        break

    if not user_input:
        continue
    if user_input.lower() in ('salir', 'exit', 'quit', 'sortir'):
        print('👋 Conversación terminada.')
        break
    if user_input.lower() == 'reset':
        reset_conversation()
        continue

    ask(user_input)

💬 Chat interactivo con el Agente Experto en ML con Scikit-Learn y TensorFlow
Comandos especiales: "salir" / "exit" para terminar | "reset" para reiniciar historial

👤 **Pregunta:** Hola

🤖 **Respuesta:**
¡Hola! ¿En qué puedo ayudarte hoy? Si tienes alguna pregunta sobre Machine Learning, no dudes en consultarme.

👋 Conversación terminada.
